# Offline JEV, Gemma, and Gemini Decision 1 comparison
Added for this snapshot. Reuses the original calculation functions, verifies all source IDs, labels and historical input hashes, and checks recomputed metrics against saved evidence. No API calls or API key required. See the README for formulas and workflow differences.

In [ ]:
from pathlib import Path
import sys, hashlib, json
root = Path.cwd() if (Path.cwd() / 'scripts').is_dir() else Path.cwd().parent
sys.path.insert(0, str(root))
from scripts.confidence_comparison import read_csv
from scripts.gemma_reasoning_recovery import source_rows
from scripts.jev_openrouter import evaluate
from scripts.evaluate_confidence import scores
import pandas as pd

source = source_rows()
report = {}
for name, file in [('JEV', 'jev_test_predictions.csv'), ('Gemma 4 score', 'gemma4_probability_test.csv'), ('Gemini Decision 1', 'gemini_decision_test_predictions.csv')]:
    rows = read_csv(root / 'data' / file)
    by_id = {int(r['source_row']): r for r in rows}
    assert len(rows) == len(by_id) == len(source)
    assert set(by_id) == {r['source_row'] for r in source}
    for email in source:
        row = by_id[email['source_row']]
        assert int(row['label']) == email['label']
        assert row['email_sha256'] == hashlib.sha256(email['email'].encode()).hexdigest()
        assert int(row['prediction']) == int(float(row['probability']) >= 0.5)
    report[name] = {**evaluate(rows), **scores([int(r['label']) for r in rows], [float(r['probability']) for r in rows])}
    saved = json.loads((root / 'data/gemini_decision_comparison.json').read_text())[name]
    for key, value in report[name].items():
        assert abs(value - saved[key]) < 1e-10, (name, key)
pd.DataFrame(report).T
